# Adaptive MEDAL: one prompt, AB-MCTS initialization, then decoding

[Open in Colab](https://colab.research.google.com/github/CooL-Legend/CSCI544-DLM/blob/adi/notebooks/gsm8k/AB_MEDAL_MCTS_Colab.ipynb)

Run this notebook on its own. Like the MEDAL demo, it loads **one LLaDA model**, shows the inline decomposition, inspects the **three initialization candidates**, and completes the selected partial state. Both notebooks share prompting, candidate displays, stage NFE and wall-clock reports, tree statistics, and final-answer scoring.

Initialization keeps the existing **Gaussian AB-MCTS GEN/CONT policy**, priors, posterior updates, immediate information gain, and cached candidate ranking. No search rollout generates a plan. An incomplete AB search remains inspectable and **skips completion and scoring**.

AB history, posterior/diversity diagnostics, artifact export, and the gated five-question GSM8K pilot are optional sections at the end, disabled by default.

## Setup — run once

Use a fresh L4/A100 Colab runtime and the LLaDA dependency profile. This demo keeps weights in the notebook kernel so you can rerun decoding without reloading the model. Setup and checks are kept here; the action cells below stay short. The private-repository checkout uses the same Colab secret/hidden-token flow as the other notebooks.

In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

REPOSITORY = "CooL-Legend/CSCI544-DLM"
BRANCH = "adi"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
IN_COLAB = Path("/content").is_dir()
REPO_DIR = Path("/content/CSCI544-DLM") if IN_COLAB else next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/dlm_project").is_dir()
)

def run_git(args, token=None):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    with tempfile.TemporaryDirectory() as tmp:
        if token:
            askpass = Path(tmp) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['COLAB_GH_TOKEN'])\n"
            )
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), COLAB_GH_TOKEN=token)
        return subprocess.run(["git", "-c", "credential.helper=", *args], env=env, capture_output=True, text=True)

def remote_git(args):
    result = run_git(args)
    if result.returncode:
        from google.colab import userdata
        try:
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if not token:
            token = getpass.getpass("GitHub token with read access to this private repository: ")
        try:
            result = run_git(args, token)
        finally:
            del token
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or "GitHub clone/update failed.")

def ensure_checkout():
    def local_git(*args):
        return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout. Choose another REPO_DIR.")
        if local_git("remote", "get-url", "origin") != REPO_URL:
            raise RuntimeError("The existing checkout belongs to a different repository.")
        if local_git("status", "--porcelain"):
            raise RuntimeError("The checkout has local changes. Save them or choose another REPO_DIR before updating.")
        fetch_specs = local_git("config", "--get-all", "remote.origin.fetch").splitlines()
        branch_ref = f"refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
        if not any(spec.lstrip("+") in (branch_ref, "refs/heads/*:refs/remotes/origin/*") for spec in fetch_specs):
            local_git("remote", "set-branches", "--add", "origin", BRANCH)
        remote_git(["-C", str(REPO_DIR), "fetch", "origin", branch_ref])
        exists = subprocess.run(["git", "-C", str(REPO_DIR), "show-ref", "--verify", "--quiet", f"refs/heads/{BRANCH}"]).returncode == 0
        if exists:
            local_git("switch", BRANCH)
        else:
            local_git("switch", "--create", BRANCH, "--track", f"origin/{BRANCH}")
        local_git("merge", "--ff-only", f"origin/{BRANCH}")
    else:
        remote_git(["clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    if local_git("branch", "--show-current") != BRANCH or local_git("rev-parse", "HEAD") != local_git("rev-parse", f"origin/{BRANCH}"):
        raise RuntimeError("The checkout differs from origin/adi. Review local commits before running experiments.")
    print(f"Code ready: {REPO_DIR} | branch: {BRANCH} | commit: {local_git('rev-parse', '--short', 'HEAD')}")

if IN_COLAB:
    ensure_checkout()
else:
    print(f"Using local checkout: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_DIR / "requirements/llada-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
                str(REPO_DIR)], check=True)
import site
for package_dir in [*site.getsitepackages(), site.getusersitepackages()]:
    site.addsitedir(package_dir)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CSCI544-DLM") if IN_COLAB else REPO_DIR / ".cache/medal"
os.environ["HF_HUB_CACHE"] = str(DRIVE_ROOT / "cache/hub")
os.environ["HF_DATASETS_CACHE"] = str(DRIVE_ROOT / "cache/datasets")
DEMONSTRATIONS_PATH = DRIVE_ROOT / "manifests/gsm8k_decomposition_train.json"

In [ ]:
import torch
from dlm_project.models.profiles import get_profile
from dlm_project.decoding.ab_medal import ABMEDALConfig
from dlm_project.decoding.ab_medal_demo import (
    run_initialization, run_completion, export_diagnostics, run_pilot,
)
from dlm_project.decoding.medal_demo import (
    DEFAULT_QUESTION, load_demo_model, prepare_demo_prompt, show_prompt,
    show_candidates, show_metrics,
)
from dlm_project.tasks.gsm8k.evaluation import score_answer
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before loading LLaDA.")
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DTYPE)

### Model source and prompt settings

The source chooser first looks for the runtime copy, then the existing Drive snapshot, then uses the pinned Hub checkpoint. Existing model snapshots and the two cached training demonstrations are reused just as in the MEDAL demo.

Change `QUESTION`, or set `PROMPT_FILE` to a UTF-8 file containing your question. Defaults remain 256 output positions, seed 1, temperature 1.0, initialization depth 20, C=3, K1=3, K2=5, the Gaussian search policy and its existing priors, and no search cap.

Set `REFERENCE_SOLUTION` to a marked answer such as `"#### 42"` for a custom question. The unchanged default bookstore question is scored against `"#### 27"`; other questions have unavailable accuracy until you supply a reference. References are used only after completion.

Set `MAX_NFE` to cap initialization forwards, including the root. AB retains its candidate-target stopping rule: a capped or exhausted search with fewer than C candidates is reported as incomplete and is never decoded. `stop_mode="budget"` is not supported by AB.

In [ ]:
model_name = get_profile("llada")["repo_id"].split("/")[-1]
stored_models = [Path("/content/models") / model_name, DRIVE_ROOT / "models" / model_name]
MODEL_PATH = next((str(path) for path in stored_models if (path / "config.json").is_file()),
                  get_profile("llada")["repo_id"])
QUESTION = DEFAULT_QUESTION
PROMPT_FILE = None
REFERENCE_SOLUTION = None  # Optional reference for a custom question.
GEN_LENGTH, SEED, TEMPERATURE = 256, 1, 1.0
MAX_NFE = None
MCTS_SETTINGS = ABMEDALConfig(
    init_tokens=20, num_candidates=3, top_k_tokens=3, top_k_actions=5,
    max_model_calls=MAX_NFE, search_policy="gaussian",
    prior_m=0.0, prior_kappa=1.0, prior_nu=1.0, prior_tau2=0.1,
)
print("Model source:", MODEL_PATH)

## 1. Load the model

Run once. Initialization and every later completion share this model.

In [ ]:
llada = load_demo_model(model_path=MODEL_PATH, device="cuda", dtype=DTYPE,
                        local_files_only=Path(MODEL_PATH).is_dir())
model, tokenizer = llada.model, llada.tokenizer

## 2. Load the prompt and show its decomposition

The full two-shot prompt is available as `prompt`; `decomposition` contains its metadata and timing. Preparation includes the fixed template, chat formatting, tokenization, and input transfer. Demonstration loading/cache bootstrap is timed separately. No extra model forward generates a plan.

In [ ]:
question = Path(PROMPT_FILE).read_text(encoding="utf-8") if PROMPT_FILE else QUESTION
prompt, prompt_ids, decomposition = prepare_demo_prompt(
    llada, question, DEMONSTRATIONS_PATH, cache_dir=os.environ["HF_DATASETS_CACHE"])
show_prompt(question)

## 3. Initialize a partial state with AB-MCTS

This cell performs search only. Gaussian GEN/CONT traversal, posterior updates, action filtering, immediate information-gain evaluation, and final candidate ranking are unchanged. Keep `initialization` to reuse the selected partial state when changing the later sampler.

The helper catches only the expected incomplete-search exception so you can inspect capped/exhausted results. It prints completion status and the stop reason. Model errors and manual interrupts propagate; the existing CLI runner used by the optional pilot retains its durable error diagnostics.

In [ ]:
initialization = run_initialization(
    model, prompt_ids, llada.mask_id, gen_length=GEN_LENGTH,
    mcts=MCTS_SETTINGS, seed=SEED, tokenizer=tokenizer,
)
completion = response = score = None

### 3.1. Top three C candidates

Each candidate lists its unmasked output positions, decoded tokens, and adjusted confidence `s`, exactly as in the MEDAL demo. `Q` is the cached selection value: the **last parent-to-child normalized entropy reduction** at a terminal candidate. The highest-Q candidate is marked. Root-to-candidate entropy reduction is displayed separately and does not change selection.

These displays read cached data and add no NFE or posterior draws. Only depth-`init_tokens` states count as candidates. If AB misses its quota, any displayed winner is provisional: completion is skipped even if one or two candidates were found. `Top 0 of 0` means no full-depth candidate was found, and AB does not decode the fallback state.

In [ ]:
show_candidates(initialization, tokenizer=tokenizer)

### 3.2. Initialization cost

The table and tree statistics use the same reporting helper as the MEDAL demo. NFE counts actual model forwards at batch size one. One root forward analyzes the fully masked output; every evaluated child adds one forward for immediate entropy reduction. Filtering uses cached logits. Selection, child creation, Gaussian draws, posterior updates, and cached candidate ranking add no forwards. Initialization performs **no full rollout**.

Stage times are exclusive: nested filtering time is moved from child evaluation to candidate generation. Backpropagation includes AB's posterior updates. Bookkeeping includes trace construction and timing overhead. CUDA boundaries are synchronized. Pipeline time is prompt + initialization + this completion; model loading, demonstration/cache setup, scoring, display, and export are separate.

Expanded parents produced evaluated children. Mean/max depth exclude the root; average branching factor is evaluated children / expanded parents. Unique states include the root and deduplicate equivalent states; evaluated children include repeated evaluations. Model evaluations equal NFE because batch size is one.

In [ ]:
metrics = show_metrics(initialization,
                       prompt_seconds=decomposition["prompt_wall_seconds"],
                       setup_seconds=decomposition["setup_wall_seconds"])

## 4. Complete the selected state

`"medal"` keeps the existing adjusted-score softmax completion. Every call receives a fresh copy of the same selected state and preserves the prompt and MCTS commitments. Rerunning this cell performs completion only; initialization is not repeated.

AB requires the full candidate quota. If initialization is incomplete, `completion` and `response` are `None`, and the following scoring cell also skips scoring. Change `DECODING_METHOD` to try another sampler after a complete initialization.

In [ ]:
DECODING_METHOD = "medal"  # Later: "my_sampler:decode" or an imported callable.
completion = run_completion(initialization, method=DECODING_METHOD, seed=SEED,
                            temperature=TEMPERATURE, tokenizer=tokenizer)
response = (tokenizer.decode(completion.token_ids[0, prompt_ids.shape[1]:].tolist(), skip_special_tokens=True)
            if completion is not None else None)
if response is not None:
    print(response)

In [ ]:
reference = REFERENCE_SOLUTION
if reference is None and question.strip() == DEFAULT_QUESTION:
    reference = "#### 27"
score = score_answer(response, reference) if completion is not None else None

In [ ]:
metrics = show_metrics(initialization, completion,
                       prompt_seconds=decomposition["prompt_wall_seconds"],
                       setup_seconds=decomposition["setup_wall_seconds"], score=score)

### Adding your sampler later

Set `DECODING_METHOD = "my_sampler:decode"`, or pass an imported function. The interface and output checks are the same as in the MEDAL demo:

```python
def decode(model, state, prompt_len, mask_id, config, *, seed, temperature, tokenizer):
    ...
```

`state` is a CPU `torch.long` tensor shaped `[1, prompt_length + output_length]`. Use the supplied counted model, fill the remaining masks, and return an object with `token_ids` of the same shape; a `trace` list is optional. The existing completion helper verifies the prompt and MCTS commitments.

Each report includes initialization **once** plus this completion's actual forwards and time. Previous sampler trials are not accumulated. `metrics` is JSON-ready. Accuracy uses only the final marked answer after generation. Missing/malformed generated answers are incorrect when a reference exists; a missing reference leaves accuracy unavailable. Incomplete AB initialization is never scored.

## 5. Why can this take a long time?

For a completed search with the default `"medal"` sampler, **total NFE = 1 root + evaluated children + remaining masks at completion**. At selected depth 20, decoding still requires **256 - 20 = 236 forwards**, committing one token per forward. An incomplete search performs no completion.

AB uses the unchanged Gaussian posterior-predictive GEN/CONT choices to grow its tree. Reaching three distinct depth-20 candidates can still require many evaluated children. No initialization cap is enabled by default, and synthetic CPU fixtures do not establish real-model target attainment or GPU speed improvements.

Every child requires inference on the full prompt-plus-output sequence and full-vocabulary entropy, margins, and filtering. Final completion samples across all legal position-token pairs. Detailed cached history, tree statistics, CPU/GPU transfers, and synchronized timing also add overhead.

The display below identifies the largest measured stages. A shorter output, different branching/depth settings, or a finite `MAX_NFE` changes the experiment. A finite cap preserves AB's strict rule: if the requested candidate quota is not met, completion and scoring are skipped.

In [ ]:
print("Largest measured stages:")
for row in sorted(metrics["stages"], key=lambda row: row["wall_seconds"], reverse=True)[:3]:
    print(f"{row['substage']}: {row['wall_seconds']:.3f}s ({row['nfe']} NFE)")

## 6. Optional AB diagnostics and artifact export

Leave `RUN_DIAGNOSTICS = False` for the MEDAL-style single-prompt flow. Enabling this section exports the existing cached initialization and the current completion; it does not rerun inference, draw from posteriors, or include earlier completion trials.

Artifacts include `config.json`, `prompt.json`, `trace.jsonl`, `history.json` with posterior updates, the full `tree.json`, `metrics.json`, an optional `completion.json`, and `ab_medal_tree.png`. The image is limited to 120 nodes/depth 20; JSON retains the full tree. Set `PRINT_HISTORY = True` to print cached search/completion events as well. Export and plotting time are outside the pipeline metrics.

Capped searches can be exported without decoding. Reports include AB decision counts, candidate diversity, and the existing full-target LLaDA gate: Gaussian policy, depth 20, three candidates, and LLaDA model identity. The gate is checked directly from initialization, independently of artifact export.

In [ ]:
RUN_DIAGNOSTICS = False
PRINT_HISTORY = False
DIAGNOSTICS_OUTPUT = DRIVE_ROOT / "results/ab_medal/single"

In [ ]:
if RUN_DIAGNOSTICS:
    directory, ab_report = export_diagnostics(
        llada, initialization, DIAGNOSTICS_OUTPUT, completion=completion,
        prompt=prompt, prompt_metadata=decomposition, question=question, score=score,
        temperature=TEMPERATURE, completion_seed=SEED, print_history=PRINT_HISTORY,
    )

In [ ]:
if RUN_DIAGNOSTICS:
    from IPython.display import Image, display
    display(Image(filename=str(directory / "ab_medal_tree.png")))

## 7. Optional five-question GSM8K AB-MCTS pilot

Leave `RUN_PILOT = False` for the single-prompt demo. The pilot requires the existing full-target Gaussian LLaDA gate and reuses the loaded model and current AB settings. The test manifest is loaded only when enabled, at the same pinned revision as the two training demonstrations. Defaults evaluate the first five test questions with the repository's per-example seed convention.

Each attempt still uses the existing AB runner, which saves search diagnostics, decodes only after reaching the full candidate quota, and uses the reference only for final-answer scoring. Paired MEDAL comparisons remain available through `scripts/compare_medal_ab_medal.py`.

The pilot reports target coverage and conditional accuracy over completed, scored attempts. Incomplete searches have unavailable correctness. Attempt records are flushed after each question; normal interrupts/errors retain completed records and an explicit partial summary. Costs cover saved attempts, including incomplete searches, and exclude work on an unfinished attempt that produced no record. Model loading is shared; dataset/cache setup is separate from per-question pipeline time.

In [ ]:
RUN_PILOT = False
PILOT_SIZE = 5
PILOT_MANIFEST = DRIVE_ROOT / "manifests/gsm8k_test.json"
PILOT_OUTPUT = DRIVE_ROOT / "results/ab_medal/pilot"

In [ ]:
if RUN_PILOT:
    pilot_directory, pilot_summary = run_pilot(
        llada, initialization, DEMONSTRATIONS_PATH, PILOT_MANIFEST, PILOT_OUTPUT,
        num_samples=PILOT_SIZE, gen_length=GEN_LENGTH, mcts=MCTS_SETTINGS, seed=SEED,
        temperature=TEMPERATURE, cache_dir=os.environ["HF_DATASETS_CACHE"],
    )